In [ ]:
import pandas as pd

In [ ]:
# Importing different datasets
path1 = r"data\results.csv"
with open(path1) as file:
    results = pd.read_csv(file)

path2 = r"data\train.csv"
with open(path2) as file:
    train = pd.read_csv(file)

path3 = r"data\test.csv"
with open(path3) as file:
    test = pd.read_csv(file)

path4 = r"data\shootouts.csv"
with open(path4) as file:
    shootouts = pd.read_csv(file)

path = r"data\confront_ranking.csv"
with open(path) as file:
    confront = pd.read_csv(file)

In [ ]:
# Having a closer look at "confront"
print(confront.shape)
print(confront.columns)
print(confront.dtypes)

In [ ]:
confront.tail()
# It does not have data of 2026 World Cup, so let's ignore it

In [ ]:
# Let's see "results"
print(results.columns)
results.tail()

In [ ]:
results.head()

In [ ]:
# Drop rows until 2006
results = results[results["date"] > "2006-01-01"]
results.head()

In [ ]:
# Drop latest matches that are not played yet
results.dropna(inplace = True)
results.tail()

In [ ]:
# selecting only World Cup mathces
results = results[results["tournament"] == "FIFA World Cup"]
print(results.shape)
results.head()

In [ ]:
# Drop irrelevant columns
results.drop(columns=["tournament", "city", "country", "neutral"], inplace=True)
results.tail()

In [ ]:
# Here knockout matches which ended up at shootout are marked as draws since home_score = away_score
# Thus I need another dataset (shootouts) to deal with it
path = r"data\shootouts.csv"
with open(path) as file:
    shootouts = pd.read_csv(file)
shootouts.columns
shootouts.tail()

In [ ]:
shootouts.iloc[678, 1], shootouts.iloc[678, -2]

In [ ]:
type(shootouts.iloc[678, 1]), type(shootouts.iloc[678, -2])

In [ ]:
shootouts.iloc[678, 1] == shootouts.iloc[678, -2]

In [ ]:
# Transform "winner" to a binary variables as the model does not accept strings

shootouts["result"] = 1 # away_team wins
for i in range(shootouts.shape[0]):
    if shootouts.iloc[i, 3] == shootouts.iloc[i, 1]: # 3 is "winner", 1 is "home_team"
        shootouts.iloc[i, -1] = 0 # -1 is "result"
    else:
        pass
shootouts.drop(columns="winner", inplace=True)
shootouts.tail()

In [ ]:
# Considering only world cups from 2006
shootouts = shootouts[shootouts["date"] > "2006-01-01"]
shootouts.head()

In [ ]:
# dropping first_team as it cointains NaN values and it is the only feature related to the shootouts.
# Thus it is useless to manage NaN values of other matches just for this feature
shootouts.drop(columns="first_shooter", inplace=True)
shootouts.head()

In [ ]:
# Merging "results" and "shootouts"
# I didn't dropped rows relative to other comptetitions in "shootouts" as with the merge they will be gone anyway
data_matches = pd.merge(results, shootouts, left_on=["date", "home_team", "away_team"], right_on=["date", "home_team", "away_team"], how="left")
data_matches.head()

In [ ]:
# testing if the merge was done correctly
data_matches[:][data_matches["date"] == "2026-06-29"]
# ok it is correct


In [ ]:
# now let's add a dummy feature to encode if the match ended up with shootout
data_matches["shootouts"] = 0
for i in range(data_matches.shape[0]):
    if data_matches.iloc[i, 3] == data_matches.iloc[i, 4]: # the two scores are even
        data_matches.iloc[i, -1] = 1 # match to shootout
    else:
        pass
data_matches.head()

In [ ]:
# testing if it is correct
data_matches[:][data_matches["date"] == "2026-06-29"]

In [ ]:
# managing "winner" for other matches ended in 90 mins
for i in range(data_matches.shape[0]):
    if pd.isnull(data_matches.iloc[i, -2]): # matche did not finished with shootout
        if data_matches.iloc[i, 3] > data_matches.iloc[i, 4]: # first team beated the second
            data_matches.iloc[i, -2] = 0 # home_team wins
        elif data_matches.iloc[i, 3] < data_matches.iloc[i, 4]:
            data_matches.iloc[i, -2] = 1 # away_team wins
        else:
            data_matches.iloc[i, -2] = 2 # draw
    else:
        pass
data_matches.tail(15)

In [ ]:
# adding a new column called "stage" 
# it is important for the training part even though it will not a feature

# enrich_results lasts till 2018, so I will use other 2 datasets for 2022 and 2026
path = r"data\enrich_results.csv"
with open(path) as file:
    enrich_results = pd.read_csv(file)
enrich_results = enrich_results[["year", "stage"]]
print(enrich_results.shape)
enrich_results.head()

In [ ]:
path = r"data\stage_2022.csv"
with open(path) as file:
    stage_2022 = pd.read_csv(file)
stage_2022 = stage_2022[["category"]]
stage_2022.head()

In [ ]:
path = r"data\stage_2026.csv"
with open(path) as file:
    stage_2026 = pd.read_csv(file)
stage_2026 = stage_2026[["stage_id"]]
stage_2026.head()

In [ ]:
# dropping all rows of "enrich_results" before 2006
enrich_results = enrich_results[enrich_results["year"] >= 2006]
print(enrich_results.head())
enrich_results.drop(columns = "year", inplace=True)

In [ ]:
# checking if there are all the matches by checking if theire length in equal to the length of results
print(enrich_results.shape[0] + stage_2022.shape[0] + stage_2026.shape[0])
data_matches.shape[0]
# deal with it after the merge to see which matches are different

In [ ]:
# Every dataset uses its own type of encoding,
# I have to make it uniformal
# 0: group stage, 1: knockouts

enrich_results.rename(columns = {"stage": "category"}, inplace=True)

enrich_results["stage"] = 1
for i in range(enrich_results.shape[0]):
    if "Group" in enrich_results.iloc[i, -2]: # -2: category
        enrich_results.iloc[i, -1] = 0 # -1: stage
    else:
        pass
print(enrich_results.iloc[:, -1].unique())
enrich_results.head()
enrich_results.drop(columns = "category", inplace=True)

In [ ]:
# same thing for "stage_2022"

stage_2022["stage"] = 1
for i in range(stage_2022.shape[0]):
    if "Group" in stage_2022.iloc[i, -2]: # -2: category
        stage_2022.iloc[i, -1] = 0 # -1: stage
    else: 
        pass
print(stage_2022.iloc[:, -1].unique())
stage_2022.head()
stage_2022.drop(columns = "category", inplace=True)


In [ ]:
# now let's do it for 2026
# dropping future matches
stage_2026 = stage_2026.iloc[:97, :]
stage_2026.tail()

In [ ]:
# check lengths
print(enrich_results.shape[0] + stage_2022.shape[0] + stage_2026.shape[0])
data_matches.shape[0]
# resolved before merge

In [ ]:
stage_2026["stage"] = 1
for i in range(stage_2026.shape[0]):
    if stage_2026.iloc[i, -2] == 1: # -2: stage_id
        stage_2026.iloc[i, -1] = 0 # -1: stage
    else: 
        pass
print(stage_2026.iloc[:, -1].unique())
stage_2026.head()
stage_2026.drop(columns = "stage_id", inplace=True)

In [ ]:
stage = pd.concat((enrich_results, stage_2022, stage_2026))
stage.head()

In [ ]:
print(type(stage)) # is a dataframe
# let's convert it into a Serie to attach it to "results"
stage = pd.Series(stage["stage"])
print(type(stage))
data_matches["stage"] = stage.values

data_matches.tail()

In [ ]:
# Let's see "train"
print(train.shape)
train.columns
# it has some new relevant features

In [ ]:
# Extracting only relevant features and basic colums
# i.e. dropping the remaining
train = train.drop(columns = ["continent", "fifa_rank_pre_tournament", "fifa_points_pre_tournament", 
                             "finalist", "semi_finalist", "quarter_finalist"])
# fifa_rank_pre_torunment could have been useful but in "confront" I already have fifa rank updated at every match
train.columns

In [ ]:
# Doing same cleaning on "test" as they are the same dataset, but for different time horizons
test = test.drop(columns = ["continent", "fifa_rank_pre_tournament", "fifa_points_pre_tournament", 
                            "finalist", "semi_finalist", "quarter_finalist"])
test.columns

In [ ]:
# Merging the three datasets
# First rebuild the entire dataset from "train" and "test"
data_teams = pd.concat([train, test])
data_teams.columns, data_teams.shape

In [ ]:
train.iloc[22, :] # Serbia
# I have modified "test" since first it was written "Serbia and Montenegro" thus the merge did not work correctly
# and I got some NaN in df

In [ ]:
test.iloc[-3, :] # Curacao
# I have modified "test" since first it was written "Cura?o" thus the merge did not work correctly
# and I got some NaN in df

In [ ]:
# Preparing datasets for the merge
# Extracting "year" from "date" in "data_matches"
data_matches[["year", "month", "day"]] = data_matches["date"].str.split("-", expand=True)
data_matches.drop(columns=["date"], inplace=True)
data_matches.columns

In [ ]:
# changing type of "year" in confront to match the one in "data_teams"
data_matches["year"] = pd.to_numeric(data_matches["year"])
data_matches.dtypes

In [ ]:
# Merging
df = pd.merge(data_matches, data_teams, left_on=["year", "home_team"], right_on = ["version", "team"], how="left").drop(columns = ["version", "team"])
df.rename(columns={
    "is_host": "home_is_host",
    "goals_scored_last_4y": "home_goals_scored_last_4y",
    "goals_received_last_4y": "home_goals_received_last_4y", 
    "wins_last_4y": "home_wins_last_4y",
    "losses_last_4y": "home_losses_last_4y",
    "draws_last_4y": "home_draws_last_4y",
    "world_cup_titles_before": "home_wc_titles_before",
    "squad_total_market_value_eur": "home_total_market_value_eur",
    "world_cup_participations_before": "home_wc_participations_before",
    "squad_avg_age": "home_avg_age",
    "groups_passed_before": "home_groups_passed_before",
    "round16_before": "home_round16_before",
    "quarterfinals_before": "home_quarterfinals_before",
    "semifinals_before": "home_semifinals_before",
    "finals_before": "home_finals_before"
}, inplace=True)

df = pd.merge(df, data_teams, left_on=["year", "away_team"], right_on = ["version", "team"], how="left").drop(columns = ["version", "team"])
df.rename(columns={
    "is_host": "away_is_host",
    "goals_scored_last_4y": "away_goals_scored_last_4y",
    "goals_received_last_4y": "away_goals_received_last_4y", 
    "wins_last_4y": "away_wins_last_4y",
    "losses_last_4y": "away_losses_last_4y",
    "draws_last_4y": "away_draws_last_4y",
    "world_cup_titles_before": "away_wc_titles_before",
    "squad_total_market_value_eur": "away_total_market_value_eur",
    "world_cup_participations_before": "away_wc_participations_before",
    "squad_avg_age": "away_avg_age",
    "groups_passed_before": "away_groups_passed_before",
    "round16_before": "away_round16_before",
    "quarterfinals_before": "away_quarterfinals_before",
    "semifinals_before": "away_semifinals_before",
    "finals_before": "away_finals_before"
}, inplace=True)
df.columns

In [ ]:
# Swapping columns to give them a better order
new_columns_list = ['year', 'month', 'day', 'home_team', 'away_team', 'stage', 'result', 'shootouts',
       'home_score', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y', 'home_losses_last_4y', 'home_draws_last_4y', 
       'home_wc_titles_before', 'home_total_market_value_eur', 'home_avg_age', 'home_wc_participations_before', 
       'home_groups_passed_before', 'home_round16_before', 'home_quarterfinals_before', 'home_semifinals_before', 'home_finals_before',
       'away_score', 'away_is_host', 'away_goals_scored_last_4y', 'away_goals_received_last_4y',
       'away_wins_last_4y', 'away_losses_last_4y', 'away_draws_last_4y', 'away_wc_titles_before', 'away_total_market_value_eur',
       'away_avg_age', 'away_wc_participations_before', 'away_groups_passed_before',
       'away_round16_before', 'away_quarterfinals_before', 'away_semifinals_before', 'away_finals_before']
df = df[new_columns_list]

In [ ]:
# checking if there are some NaN in "df"
container = list()
for j in range(df.shape[0]):
    count = 0 # counts only the first column so "container" is not too much long
    for i in df.columns:
        if count == 0:
            if df[i].isna().iloc[j]:
                count += 1
                container.append(f"Missing value at row {j} column {i}")
            else:
                pass
        else:
            pass
print(container)
print(len(container))

In [ ]:
# A list of matches with some NaN at first detected by the code above,
# but then corrected directly modifing "train" and "test"

In [ ]:
df.iloc[7, :]

In [ ]:
df.iloc[20, :]

In [ ]:
df.iloc[37, :]

In [ ]:
df.loc[328, :]

In [ ]:
df.loc[353, :]

In [ ]:
df.iloc[376, :]

In [ ]:
# last look at the data before exporting
print(df.shape)
df.columns

In [ ]:
df.to_csv(r"data\dataset.csv", index=False)